In [29]:
#1. Retrieval: gives results from vector store based on query
# def retrieve: Retrieve relevant documents from the vector store in form of a list of dictionaries 
# query is tranformed into an embedding and then used to search the vector store for similar documents

import sys
sys.path.insert(0, '../')

from typing import List, Dict, Any
from src.embedding_manager import EmbeddingManager
from src.vector_store import FaissVectorStore

class RAGRetriever:
    def __init__(self, vector_store: FaissVectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        try:
            # Generate embedding for the query
            query_embeddings = self.embedding_manager.generate_embeddings([query])
            query_embedding = query_embeddings[0]
            
            # Search the vector store - returns list of tuples (metadata, distance)
            results = self.vector_store.search(query_embedding, top_k=top_k)
            retrieved_docs = []
            
            for rank, (metadata, distance) in enumerate(results, 1):
                similarity_score = distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': rank,
                        'content': metadata.get('content', ''),
                        'source': metadata.get('source', ''),
                        'similarity_score': float(similarity_score),
                        'metadata': metadata,
                        'rank': rank
                    })
            
            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}")
            else:
                print(f"No documents retrieved above the score threshold of {score_threshold}")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            import traceback
            traceback.print_exc()
            return []

In [30]:
#2. Initialize vector store and embedding manager (load from disk)

try:
    embedding_manager = EmbeddingManager()
    print("EmbeddingManager initialized successfully")
except Exception as e:
    print(f"Failed to initialize EmbeddingManager: {e}")
    embedding_manager = None

try:
    vector_store = FaissVectorStore(embedding_dim=embedding_manager.get_embedding_dimension())
    print(f"Vector store loaded successfully with {len(vector_store.id_to_metadata)} chunks")
except Exception as e:
    print(f"Failed to initialize Vector Store: {e}")
    vector_store = None

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 69596.54it/s]


Loaded embedding model: BAAI/bge-m3. Embedding dimension: 1024
EmbeddingManager initialized successfully
Loaded index with 593 chunks
Vector store loaded successfully with 593 chunks


In [31]:
# 3. Initialize RAGRetriever with instances from ingestion.ipynb

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"
results = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

for doc in results:
    
    print(f"\nRank {doc['rank']}: {doc['similarity_score']:.4f}")
    print(f"Source: {doc['source']}")
    print(f"Content preview: {doc['content'][:200]}...")


Retrieving documents for query: 'What is the economic contribution of the Great Barrier Reef?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.05it/s]


Retrieved 5 documents above the score threshold of 0.0

Rank 1: 0.6049
Source: Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf
Content preview: and US$293.5 million (ie, between about A$120 million and A$375 million) by.21 Given Access Economics’ estimates for the-06 direct economic contribution of tourism to the GBRCA as a whole, which inclu...

Rank 2: 0.5912
Source: {'source': 'Access Economics 2007 Economic contribution of Great Barrier Reef Marine Park 2005-2006 Kopie.pdf', 'author': 'Great Barrier Reef Marine Park Authority', 'year': 2007, 'description': 'A comprehensive study on the economic contribution of the Great Barrier Reef Marine Park for the years 2005-2006.'}
Content preview: than implied by average rates of global warming, especially given the small increases in maximum temperatures needed to induce it. For the Reef, recovery speed and resilience are the immediate policy ...

Rank 3: 0.5841
Source: Access Economics 2007

In [32]:
# 3b. Debug: Check raw similarity scores

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"

# Get results with NO threshold filtering
results_no_filter = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

print("Raw similarity scores (without threshold filter):")
for doc in results_no_filter:
    print(f"  Rank {doc['rank']}: {doc['similarity_score']:.4f}")


Retrieving documents for query: 'What is the economic contribution of the Great Barrier Reef?' with top_k=5 and score_threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.02it/s]

Retrieved 5 documents above the score threshold of 0.0
Raw similarity scores (without threshold filter):
  Rank 1: 0.6049
  Rank 2: 0.5912
  Rank 3: 0.5841
  Rank 4: 0.5821
  Rank 5: 0.5814


In [33]:
#4. LLM integration with Ollama

from langchain_ollama import OllamaLLM
import os
from dotenv import load_dotenv
load_dotenv()

#temperature defines how "creative" the model's answers will be --> low value for solid and factual answers
llm = OllamaLLM(model="llama3", temperature=0.2)


In [34]:
#5. Simple RAG function that combines retrieval and generation

def rag_answer(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, score_threshold: float = 0.55):

    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=score_threshold)
    
    if not results:
        return "No relevant documents found to answer the question."

    context = "\n".join([doc['content'] for doc in results])
    
    prompt = f"""Use the following context to answer the question concisely and factually.

        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke(prompt)
    return response

In [37]:
#6. Test the RAG function
answer=rag_answer("What are the ecological benefits of the Great Barrier Reef?", rag_retriever, llm, score_threshold=0.6)
print(answer)

Retrieving documents for query: 'What are the ecological benefits of the Great Barrier Reef?' with top_k=5 and score_threshold=0.6


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.56it/s]

No documents retrieved above the score threshold of 0.6
No relevant documents found to answer the question.


In [ ]:
#7. enhance : returns answer but also source, confidence score and optionally the context used for answering

def rag_answer_enhanced(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': "No relevant documents found to answer the question.", 'sources': [], 'confidence': 0.0, 'context': []}
    #prepare context and sources
    context = "\n".join([doc['content'] for doc in results])
    sources = [{
            'source': doc['source'],
            'content': doc['content'],
            'score': doc['similarity_score'],
            'preview': doc['content'][:200] + "..."
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)

    # generate answer
    prompt = f"""Use the following context to answer the question concisely and factually.
    {context}"""
    answer = llm.generate(prompt)

    output = {
        'answer': answer,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_answer_enhanced("What are the ecological benefits of the Great Barrier Reef?", rag_retriever, llm, top_k=3, min_score=0.6, return_context=True)
print("Answer: ", result['answer'])
print("Sources: ", result['sources'])
print("Confidence Score: ", result['confidence'])
print("Context used for answering: ", result.get('context', 'No context returned')[:200])